# Re-analisis Prediksi Hepatitis dengan Data Mining
**Pengaruh urutan praproses terhadap kinerja prediksi pada dua dataset UCI (HCV dan Hepatitis)**

Notebook ini mengikuti prosedur 14 langkah di proposal. Tujuannya bukan mencari model terbaik, tetapi menjalankan satu alur kerja yang dijelaskan langkah demi langkah, lalu **mengukur bagaimana skor berubah ketika urutan praproses terhadap pembagian data diubah** (V1-V4), dan menguji kestabilannya. Apa pun hasilnya dilaporkan apa adanya.

**Peta langkah protokol ke bagian notebook**

| Langkah | Bagian |
|---|---|
| 1. Unduh data, catat tanggal dan checksum | Bagian 1 |
| 2. Pemeriksaan awal tanpa transformasi | Bagian 2 |
| 3. Definisi tugas dan kelas positif | Bagian 3 |
| 4. Pembagian data (repeated stratified 5-fold x 10) | Bagian 4 |
| 5-9. Imputasi, encoding/penskalaan, penyeimbangan, seleksi fitur, tuning (semua di dalam lipatan) | Bagian 5 |
| 10. Evaluasi metrik | Bagian 6 |
| 11. Variasi urutan praproses V1-V4 (+ V2b, V4b) | Bagian 7 |
| 12. Selisih berpasangan, *corrected resampled t-test*, koreksi uji ganda | Bagian 9 |
| 13. Uji sensitivitas | Bagian 10 |
| 14. Dokumentasi dan reproduksibilitas | Bagian 11 |

**Cara menjalankan:** simpan notebook ini di folder `hepatitis` (sejajar dengan subfolder `hcv+data` dan `hepatitis`), lalu jalankan sel dari atas ke bawah. Jalankan dulu dengan `FAST_MODE = True` untuk uji coba (cepat), baru ubah ke `False` untuk hasil final.

**Catatan jujur tentang keterbatasan:** data sangat kecil (UCI HCV: 75 pasien; UCI Hepatitis: 32 kasus meninggal), sehingga hasil per lipatan bervariasi besar. Karena lipatan pada *repeated CV* saling tumpang tindih, uji statistik memakai *corrected resampled t-test* (Nadeau & Bengio, 2003) yang memperhitungkan tumpang tindih tersebut. Tafsirkan dengan hati-hati dan jangan membuat klaim klinis.

**Perubahan pada versi 2 (9 Oktober 2026)**
1. **V2b dan V4b (baru).** Pada V2 dan V4 lama, data uji ikut berisi titik sintetis SMOTE sehingga proporsi kelas data uji berubah menjadi 50:50. F1 dan PR-AUC sangat dipengaruhi proporsi ini, jadi selisih V2 mencampur dua efek. V2b dan V4b memakai **pembagian yang sama persis dengan P-C**, menambahkan titik sintetis (yang dibuat dari seluruh data, termasuk baris uji) ke data latih, dan menilai **hanya baris asli** di data uji. Dengan begitu: `V2 - V2b` = efek komposisi data uji, `V2b - P-C` = kebocoran sungguhan, dan V2b/V4b bisa diuji berpasangan.
2. **Uji statistik.** Bootstrap dan Wilcoxon diganti dengan *corrected resampled t-test* untuk semua perbandingan berpasangan (V1, V2b, V3, V4b), ditambah koreksi uji ganda Holm dan Benjamini-Hochberg. V2 dan V4 (tidak berpasangan) dilaporkan deskriptif saja.
3. **Path data** dibuat relatif terhadap folder notebook.

## Bagian 0. Konfigurasi dan pustaka

In [ ]:
# Instal sekali (hapus tanda # jika perlu), lalu restart kernel:
# !pip install pandas numpy scikit-learn imbalanced-learn xgboost scipy matplotlib

import os, sys, time, platform, hashlib, warnings, datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from importlib.metadata import version, PackageNotFoundError

from sklearn.base import clone
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import (RepeatedStratifiedKFold, StratifiedKFold,
                                     GridSearchCV, ParameterGrid, cross_validate)
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import make_scorer, precision_score, get_scorer
from joblib import Parallel, delayed
from imblearn.pipeline import Pipeline          # Pipeline yang mendukung SMOTE di dalam lipatan
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

# ---------------- KONFIGURASI (ubah di sini saja) ----------------
FAST_MODE   = False      # True = uji coba cepat (ulangan sedikit, pohon sedikit). False = hasil final.
SEED        = 42         # seed utama; dicatat di laporan
N_SPLITS    = 5          # k pada stratified k-fold
N_REPEATS   = 2 if FAST_MODE else 10
INNER_FOLDS = 3          # validasi dalam untuk tuning
K_FEATURES  = 8          # jumlah fitur pada SelectKBest
N_TREES     = 50 if FAST_MODE else 200
N_BOOT      = 500 if FAST_MODE else 5000
DATA_DIR    = "data"
OUT_DIR     = "results"
MODELS      = ["NB", "DT", "KNN", "RF", "GB", "XGB"]
SENS_MODELS = ["NB", "DT", "GB"] if FAST_MODE else MODELS
os.makedirs(OUT_DIR, exist_ok=True)

SCORING = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": "recall",            # = sensitivitas pada kelas positif
    "f1": "f1",
    "roc_auc": "roc_auc",
    "average_precision": "average_precision",   # PR-AUC
}
METRICS = list(SCORING)
print("Python", platform.python_version(), "| FAST_MODE =", FAST_MODE,
      "| N_REPEATS =", N_REPEATS, "| tanggal:", datetime.date.today())

## Bagian 1. Langkah 1: Unduh data, catat tanggal akses, simpan checksum
Unduh dari UCI Machine Learning Repository:
- **HCV data**: https://archive.ics.uci.edu/dataset/571/hcv+data (file `hcvdat0.csv`)
- **Hepatitis**: https://archive.ics.uci.edu/dataset/46/hepatitis (file `hepatitis.data`)

Data sudah disertakan di folder `data/`; tanggal akses asli adalah 8 Oktober 2026. Checksum SHA-256 dipakai agar pembaca bisa memastikan file yang sama. Path dibuat relatif agar notebook bisa dijalankan di komputer mana pun.

In [ ]:
TANGGAL_AKSES = "2026-10-08"   # tanggal unduh dari UCI

# Path relatif terhadap folder utama repositori
FILES = {"HCV": os.path.join("data", "hcv", "hcvdat0.csv"),
         "Hepatitis": os.path.join("data", "hepatitis", "hepatitis.data")}
for v in FILES.values():
    assert os.path.exists(v), f"File tidak ditemukan: {v}. Jalankan notebook dari folder utama repositori."

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for blok in iter(lambda: f.read(8192), b""):
            h.update(blok)
    return h.hexdigest()

cek = pd.DataFrame([{"dataset": k, "file": v, "ukuran_byte": os.path.getsize(v),
                     "sha256": sha256(v), "tanggal_akses": TANGGAL_AKSES}
                    for k, v in FILES.items()])
cek.to_csv(os.path.join(OUT_DIR, "checksum_data.csv"), index=False)
cek

## Bagian 2. Langkah 2: Pemeriksaan awal (tanpa transformasi apa pun)
Pada tahap ini data hanya **dibaca dan dilaporkan**. Tidak ada imputasi, penghapusan baris, atau penskalaan.

In [ ]:
HEP_COLS = ["Class", "AGE", "SEX", "STEROID", "ANTIVIRALS", "FATIGUE", "MALAISE", "ANOREXIA",
            "LIVER_BIG", "LIVER_FIRM", "SPLEEN_PALPABLE", "SPIDERS", "ASCITES", "VARICES",
            "BILIRUBIN", "ALK_PHOSPHATE", "SGOT", "ALBUMIN", "PROTIME", "HISTOLOGY"]

raw_hcv = pd.read_csv(FILES["HCV"])
raw_hcv = raw_hcv.rename(columns={raw_hcv.columns[0]: "X"})          # kolom ID tanpa nama
raw_hep = pd.read_csv(FILES["Hepatitis"],
                      header=None, names=HEP_COLS, na_values="?")

def periksa(df, nama, kolom_target):
    print("=" * 70)
    print(nama, "| dimensi:", df.shape)
    print("Baris duplikat:", int(df.duplicated().sum()))
    miss = df.isna().sum()
    print("Missing value per kolom:", miss[miss > 0].to_dict() or "tidak ada")
    print("Baris dengan >=1 missing:", int(df.isna().any(axis=1).sum()))
    print("Distribusi target:", df[kolom_target].value_counts().to_dict())
    print(df.dtypes.value_counts().to_dict())

periksa(raw_hcv, "UCI HCV", "Category")
periksa(raw_hep, "UCI Hepatitis", "Class")

## Bagian 3. Langkah 3: Definisi tugas dan kelas positif
- **UCI HCV:** kelas positif (1) = pasien Hepatitis C (hepatitis, fibrosis, sirosis). Kelas negatif (0) = donor darah, termasuk 7 *suspect donor*.
- **UCI Hepatitis:** kelas positif (1) = meninggal (`Class == 1`). Kelas negatif (0) = selamat.

Kolom `X` (ID) dibuang. Jenis kelamin HCV di-encode (m=1, f=0). Pada UCI Hepatitis, nilai 1/2 pada fitur biner dibiarkan apa adanya karena model berbasis pohon, NB, dan KNN tidak terganggu oleh pengkodean ini.

In [ ]:
def siapkan_hcv(df):
    y = (~df["Category"].str.startswith("0")).astype(int)          # 0=Blood Donor & 0s=suspect -> 0
    X = df.drop(columns=["X", "Category"]).copy()
    X["Sex"] = (X["Sex"] == "m").astype(int)
    return X.reset_index(drop=True), y.reset_index(drop=True)

def siapkan_hep(df):
    y = (df["Class"] == 1).astype(int)                             # 1 = DIE = positif
    X = df.drop(columns=["Class"]).copy()
    return X.reset_index(drop=True), y.reset_index(drop=True)

DATASETS = {"HCV": siapkan_hcv(raw_hcv), "Hepatitis": siapkan_hep(raw_hep)}

baris = []
for nama, (X, y) in DATASETS.items():
    maj = max(y.mean(), 1 - y.mean())
    baris.append({"dataset": nama, "n": len(y), "fitur": X.shape[1], "positif": int(y.sum()),
                  "negatif": int((1 - y).sum()), "persen_positif": round(100 * y.mean(), 1),
                  "baseline_akurasi_mayoritas": round(100 * maj, 1)})
pd.DataFrame(baris)

## Bagian 4. Langkah 4: Skema pembagian data
*Repeated stratified 5-fold* dengan 10 ulangan = 50 pembagian per model. Seed tetap. Pada setiap pembagian, **semua langkah 5-9 dipelajari hanya dari data latih**. Pembagian yang sama dipakai oleh alur utama (P-A/B/C) dan variasi yang datanya tidak berubah ukuran, sehingga selisihnya bisa dipasangkan.

In [ ]:
def buat_cv(seed=SEED):
    return RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=seed)

print("Jumlah pembagian per model:", N_SPLITS * N_REPEATS)

## Bagian 5. Langkah 5-9: Alur utama (praproses di dalam lipatan)
Urutan di dalam setiap lipatan: **imputasi -> (penskalaan untuk KNN) -> seleksi fitur -> (SMOTE) -> model**, dengan tuning hiperparameter memakai validasi dalam (*inner CV*) pada data latih saja.

Model yang dibandingkan: Naive Bayes (NB), Decision Tree (DT), KNN, Random Forest (RF), **Gradient Boosting (GB, scikit-learn)**, dan XGBoost (XGB). Semuanya dituning dengan kisi hiperparameter kecil yang sama-sama adil di dalam lipatan, sehingga perbandingan GB lawan DT tidak berat sebelah.

Skenario penyeimbangan kelas: **A** tanpa penanganan, **B** *class weight* (hanya untuk DT, RF, XGB; NB, KNN, dan Gradient Boosting scikit-learn tidak punya fitur ini), **C** SMOTE.

In [ ]:
class XGBSeimbang(XGBClassifier):
    # class weight untuk XGBoost: rasio negatif/positif dihitung dari data latih lipatan saja
    def fit(self, X, y, **kw):
        y_arr = np.asarray(y)
        self.set_params(scale_pos_weight=(y_arr == 0).sum() / max((y_arr == 1).sum(), 1))
        return super().fit(X, y, **kw)

def klasifier(nama, balance, seed):
    cw = "balanced" if balance == "class_weight" else None
    if nama == "NB":
        return GaussianNB()
    if nama == "DT":
        return DecisionTreeClassifier(class_weight=cw, random_state=seed)
    if nama == "KNN":
        return KNeighborsClassifier()
    if nama == "RF":
        return RandomForestClassifier(n_estimators=N_TREES, class_weight=cw, random_state=seed, n_jobs=1)
    if nama == "GB":
        return GradientBoostingClassifier(n_estimators=N_TREES, random_state=seed)
    if nama == "XGB":
        kelas = XGBSeimbang if balance == "class_weight" else XGBClassifier
        return kelas(n_estimators=N_TREES, eval_metric="logloss", random_state=seed, n_jobs=1, verbosity=0)
    raise ValueError(nama)

GRIDS = {
    "NB":  {"clf__var_smoothing": [1e-9, 1e-7, 1e-5]},
    "DT":  {"clf__max_depth": [3, 5, None], "clf__min_samples_leaf": [1, 5]},
    "KNN": {"clf__n_neighbors": [3, 5, 9]},
    "RF":  {"clf__max_depth": [None, 5], "clf__min_samples_leaf": [1, 3]},
    "GB":  {"clf__max_depth": [2, 3], "clf__learning_rate": [0.05, 0.2]},
    "XGB": {"clf__max_depth": [2, 4], "clf__learning_rate": [0.05, 0.2]},
}
BERLAKU = {"none": set(MODELS), "class_weight": {"DT", "RF", "XGB"}, "smote": set(MODELS)}

def buat_estimator(nama, balance, imputer="median", use_fs=True, n_fitur=None, tune=True, seed=SEED):
    # imputer: "median", "knn", atau "none" (data sudah tanpa NaN)
    langkah = []
    if imputer == "median":
        langkah.append(("imp", SimpleImputer(strategy="median")))
    elif imputer == "knn":
        langkah.append(("imp", KNNImputer(n_neighbors=5)))
    if nama == "KNN":
        langkah.append(("scale", StandardScaler()))
    if use_fs:
        langkah.append(("fs", SelectKBest(f_classif, k=min(K_FEATURES, n_fitur))))
    if balance == "smote":
        langkah.append(("smote", SMOTE(random_state=seed, k_neighbors=3)))
    langkah.append(("clf", klasifier(nama, balance, seed)))
    pipa = Pipeline(langkah)
    if tune:
        return GridSearchCV(pipa, GRIDS[nama], scoring="average_precision",
                            cv=StratifiedKFold(INNER_FOLDS, shuffle=True, random_state=seed), n_jobs=1,
                            error_score="raise")
    return pipa

def rapikan(res, dataset, model, prosedur, config="utama"):
    d = pd.DataFrame({m: res[f"test_{m}"] for m in METRICS})
    d.insert(0, "split", np.arange(len(d)))
    d.insert(0, "config", config)
    d.insert(0, "prosedur", prosedur)
    d.insert(0, "model", model)
    d.insert(0, "dataset", dataset)
    return d

NAMA_PIPA = {"none": "P-A", "class_weight": "P-B", "smote": "P-C"}

def jalankan_pipa(X, y, dataset, nama, balance, imputer="median", seed=SEED, config="utama"):
    est = buat_estimator(nama, balance, imputer, use_fs=True, n_fitur=X.shape[1], seed=seed)
    res = cross_validate(est, X, y, cv=buat_cv(seed), scoring=SCORING, n_jobs=-1, error_score="raise")
    return rapikan(res, dataset, nama, NAMA_PIPA[balance], config)

## Bagian 6. Langkah 10: Evaluasi alur utama (P-A, P-B, P-C)
Metrik: akurasi, *balanced accuracy*, presisi, sensitivitas (recall), F1, ROC-AUC, PR-AUC. Akurasi selalu dibaca bersama baseline kelas mayoritas (Bagian 3).

In [ ]:
def jalankan_semua_pipa(models=MODELS, imputer="median", seed=SEED, config="utama", balances=("none", "class_weight", "smote")):
    hasil = []
    for ds, (X, y) in DATASETS.items():
        for nama in models:
            for b in balances:
                if nama not in BERLAKU[b]:
                    continue
                t0 = time.time()
                hasil.append(jalankan_pipa(X, y, ds, nama, b, imputer, seed, config))
                print(f"[{ds}] {nama:3s} {NAMA_PIPA[b]}  ({time.time() - t0:.0f} dtk)")
    return pd.concat(hasil, ignore_index=True)

hasil_pipa = jalankan_semua_pipa()
hasil_pipa.to_csv(os.path.join(OUT_DIR, "hasil_pipa_mentah.csv"), index=False)

In [ ]:
def ringkas(df, metrik=("accuracy", "recall", "precision", "f1", "average_precision", "roc_auc")):
    g = df.groupby(["dataset", "model", "prosedur"])[list(metrik)]
    rata, sd = g.mean(), g.std()
    out = rata.round(3).astype(str) + " ± " + sd.round(3).astype(str)
    return out

ringkas(hasil_pipa)

## Bagian 7. Langkah 11: Variasi urutan praproses (V1-V4, V2b, V4b)
Tujuan bagian ini: mengetahui seberapa besar skor berubah bila **satu** langkah praproses dipelajari dari seluruh data sebelum pembagian, sedangkan langkah lain tetap sama. Acuan perbandingan adalah **P-C** (SMOTE di dalam lipatan).

| Kode | Perubahan urutan | Cara kerja | Data uji | Berpasangan dengan P-C? |
|---|---|---|---|---|
| **V1** | Imputasi sebelum pembagian | Imputer dipelajari dari seluruh data, langkah lain sama dengan P-C | baris asli | ya |
| **V2** | SMOTE sebelum pembagian (cara yang umum di literatur) | Imputasi seluruh data, SMOTE seluruh data, lalu CV pada data hasil SMOTE | baris asli **+ titik sintetis** (kelas ~50:50) | tidak |
| **V2b** | SMOTE sebelum pembagian, dinilai pada baris asli | Titik sintetis dibuat dari seluruh data (termasuk baris uji) lalu ditambahkan ke data latih setiap lipatan; pembagian sama dengan P-C | **hanya baris asli** | ya |
| **V3** | Seleksi fitur pada seluruh data | Fitur dipilih memakai seluruh data (dan label), lalu alur P-C dijalankan pada fitur terpilih | baris asli | ya |
| **V4** | Gabungan | V1 + V2 + V3 + hiperparameter dipilih berdasarkan skor data uji | baris asli + titik sintetis | tidak |
| **V4b** | Gabungan, dinilai pada baris asli | V1 + V2b + V3 + hiperparameter dipilih berdasarkan skor data uji | **hanya baris asli** | ya |

**Cara membaca V2 dan V2b.** Selisih `V2 - P-C` adalah angka yang biasanya muncul bila SMOTE dilakukan sebelum pembagian. Selisih itu dipecah menjadi dua:
- `V2b - P-C` = **kebocoran sungguhan**: model dilatih dengan titik sintetis yang sebagian dibuat dari baris uji.
- `V2 - V2b` = **efek komposisi data uji**: data uji berisi titik sintetis dan proporsi kelasnya berubah, sehingga F1 dan PR-AUC naik walau model tidak lebih baik.

Catatan: pada V2 data latih berisi sekitar 80% titik sintetis (sisanya masuk data uji), sedangkan pada V2b seluruh titik sintetis masuk data latih. Perbedaan kecil ini disebutkan di keterbatasan.

In [ ]:
def impute_global(X, imputer="median"):
    imp = SimpleImputer(strategy="median") if imputer == "median" else KNNImputer(n_neighbors=5)
    return pd.DataFrame(imp.fit_transform(X), columns=X.columns)       # dipelajari dari seluruh baris (sebelum pembagian)

def smote_global(Xi, y, seed):
    Xr, yr = SMOTE(random_state=seed, k_neighbors=3).fit_resample(Xi, y)   # dilakukan pada seluruh data sebelum pembagian
    Xr, yr = np.asarray(Xr), np.asarray(yr)
    n = len(y)
    # imblearn menaruh baris asli di urutan awal, titik sintetis di belakang. Dicek agar penandaan baris asli pasti benar.
    assert np.allclose(Xr[:n], np.asarray(Xi, dtype=float)) and (yr[:n] == np.asarray(y)).all(), \
        "Urutan keluaran SMOTE tidak sesuai: baris asli tidak berada di awal."
    assert (yr[n:] == 1).all(), "Titik sintetis seharusnya semuanya kelas positif."
    return Xr, yr

SCORERS = {m: (get_scorer(s) if isinstance(s, str) else s) for m, s in SCORING.items()}

def _skor(est, X, y):
    return {f"test_{m}": SCORERS[m](est, X, y) for m in METRICS}

def _satu_lipatan_b(est, X_latih, y_latih, X_uji, y_uji):
    e = clone(est).fit(X_latih, y_latih)
    return _skor(e, X_uji, y_uji)

def cv_tambah_sintetis(est, Xi, y, X_sintetis, cv, kolom=None):
    """Pembagian dibuat dari baris asli (sama dengan P-C). Semua titik sintetis ditambahkan ke data latih;
    data uji hanya berisi baris asli. kolom: indeks fitur terpilih (untuk V4b)."""
    Xi = np.asarray(Xi, dtype=float); y = np.asarray(y)
    if kolom is not None:
        Xi, X_sintetis = Xi[:, kolom], X_sintetis[:, kolom]
    y_sint = np.ones(len(X_sintetis), dtype=int)
    tugas = (delayed(_satu_lipatan_b)(est, np.vstack([Xi[tr], X_sintetis]), np.concatenate([y[tr], y_sint]),
                                      Xi[te], y[te]) for tr, te in cv.split(Xi, y))
    hasil = Parallel(n_jobs=-1)(tugas)
    return {k: np.array([h[k] for h in hasil]) for k in hasil[0]}

def jalankan_variasi(X, y, dataset, nama, skenario, imputer="median", seed=SEED, config="utama"):
    cv = buat_cv(seed)
    n = len(y)
    if skenario == "V1":
        Xi = impute_global(X, imputer)
        est = buat_estimator(nama, "smote", "none", use_fs=True, n_fitur=Xi.shape[1], seed=seed)
        res = cross_validate(est, Xi, y, cv=cv, scoring=SCORING, n_jobs=-1, error_score="raise")
    elif skenario == "V2":
        Xr, yr = smote_global(impute_global(X, imputer), y, seed)
        est = buat_estimator(nama, "none", "none", use_fs=True, n_fitur=Xr.shape[1], seed=seed)
        res = cross_validate(est, Xr, yr, cv=cv, scoring=SCORING, n_jobs=-1, error_score="raise")
    elif skenario == "V2b":
        Xi = impute_global(X, imputer)
        Xr, yr = smote_global(Xi, y, seed)
        est = buat_estimator(nama, "none", "none", use_fs=True, n_fitur=Xi.shape[1], seed=seed)
        res = cv_tambah_sintetis(est, Xi, y, Xr[n:], cv)
    elif skenario == "V3":
        Xi = impute_global(X, imputer)
        terpilih = SelectKBest(f_classif, k=min(K_FEATURES, Xi.shape[1])).fit(Xi, y).get_support()   # fitur dipilih dari seluruh data
        Xs = X.loc[:, terpilih]
        est = buat_estimator(nama, "smote", imputer, use_fs=False, seed=seed)
        res = cross_validate(est, Xs, y, cv=cv, scoring=SCORING, n_jobs=-1, error_score="raise")
    elif skenario in ("V4", "V4b"):
        Xi = impute_global(X, imputer)
        Xr, yr = smote_global(Xi, y, seed)
        terpilih = SelectKBest(f_classif, k=min(K_FEATURES, Xr.shape[1])).fit(Xr, yr).get_support()  # fitur dipilih dari seluruh data
        kolom = np.flatnonzero(terpilih)
        terbaik = None
        for params in ParameterGrid(GRIDS[nama]):               # hiperparameter dipilih berdasarkan skor data uji
            est = buat_estimator(nama, "none", "none", use_fs=False, tune=False, seed=seed).set_params(**params)
            if skenario == "V4":
                r = cross_validate(est, Xr[:, kolom], yr, cv=cv, scoring=SCORING, n_jobs=-1, error_score="raise")
            else:
                r = cv_tambah_sintetis(est, Xi, y, Xr[n:], cv, kolom=kolom)
            skor = r["test_average_precision"].mean()
            if terbaik is None or skor > terbaik[0]:
                terbaik = (skor, r)
        res = terbaik[1]
    else:
        raise ValueError(skenario)
    return rapikan(res, dataset, nama, skenario, config)

SEMUA_VARIASI = ("V1", "V2", "V2b", "V3", "V4", "V4b")

def jalankan_semua_variasi(models=MODELS, skenario=SEMUA_VARIASI, imputer="median", seed=SEED, config="utama"):
    hasil = []
    for ds, (X, y) in DATASETS.items():
        for nama in models:
            for s in skenario:
                t0 = time.time()
                hasil.append(jalankan_variasi(X, y, ds, nama, s, imputer, seed, config))
                print(f"[{ds}] {nama:3s} {s:3s}  ({time.time() - t0:.0f} dtk)")
    return pd.concat(hasil, ignore_index=True)

hasil_variasi = jalankan_semua_variasi()
hasil_variasi.to_csv(os.path.join(OUT_DIR, "hasil_variasi_mentah.csv"), index=False)

## Bagian 8. Ringkasan semua prosedur

In [ ]:
semua = pd.concat([hasil_pipa, hasil_variasi], ignore_index=True)
semua.to_csv(os.path.join(OUT_DIR, "hasil_semua_mentah.csv"), index=False)
assert not semua[METRICS].isna().any().any(), "Ada metrik kosong (NaN): periksa log, jangan lanjut ke analisis."

urutan = ["P-A", "P-B", "P-C", "V1", "V2", "V2b", "V3", "V4", "V4b"]
rata = (semua.groupby(["dataset", "model", "prosedur"])[["accuracy", "recall", "f1", "average_precision"]]
        .mean().round(3))
tabel = rata.reset_index().pivot_table(index=["dataset", "model"], columns="prosedur",
                                       values="f1").reindex(columns=urutan).round(3)
print("Rata-rata F1 per prosedur:")
tabel

## Bagian 9. Langkah 12: Selisih (variasi dikurangi P-C), *corrected resampled t-test*, koreksi uji ganda
Selisih positif = variasi memberi skor **lebih tinggi** daripada P-C; selisih nol atau negatif juga dilaporkan.

**Mengapa uji t biasa atau Wilcoxon tidak cukup.** Pada *repeated 5-fold CV*, setiap baris data dipakai sebagai data latih di banyak lipatan, sehingga 50 skor per model tidak saling bebas. Uji yang menganggapnya bebas memberi galat baku yang terlalu kecil dan p-value yang terlalu kecil.

**Corrected resampled t-test** (Nadeau & Bengio, 2003; Bouckaert & Frank, 2004). Untuk selisih per lipatan $d_j$, $j = 1..J$ ($J = k \times r = 50$):

$$t = \frac{\bar d}{\sqrt{\left(\frac{1}{J} + \frac{n_{uji}}{n_{latih}}\right)\hat\sigma^2_d}}, \qquad df = J - 1$$

dengan $n_{uji}/n_{latih} = 1/(k-1) = 0{,}25$ pada 5-fold. Suku $n_{uji}/n_{latih}$ adalah koreksi untuk tumpang tindih. Interval kepercayaan 95% memakai galat baku yang sama: $\bar d \pm t_{0.975,\,df} \cdot SE$.

**Koreksi uji ganda.** Ada banyak uji (dataset × model × variasi × metrik). Laporkan p-value Holm (mengendalikan *family-wise error*, lebih ketat) dan Benjamini-Hochberg (mengendalikan *false discovery rate*). Keluarga uji dihitung **per metrik**, karena tiap metrik menjawab pertanyaan yang berbeda.

**V2 dan V4** tidak berpasangan (data ujinya berbeda), sehingga tidak diuji. Selisihnya dilaporkan deskriptif dan dipecah dengan V2b/V4b (lihat Bagian 7).

In [ ]:
def ttest_terkoreksi(d, n_splits=N_SPLITS):
    """Corrected resampled t-test (Nadeau & Bengio 2003) untuk selisih per lipatan dari repeated k-fold CV."""
    d = np.asarray(d, dtype=float)
    J = len(d)
    rasio = 1.0 / (n_splits - 1)                     # n_uji / n_latih
    rata, var = d.mean(), d.var(ddof=1)
    se = np.sqrt((1.0 / J + rasio) * var)
    tkrit = stats.t.ppf(0.975, J - 1)
    if se == 0:                                      # semua selisih identik
        return rata, rata, rata, np.nan, np.nan
    t = rata / se
    p = 2 * stats.t.sf(abs(t), J - 1)
    return rata, rata - tkrit * se, rata + tkrit * se, t, p

def holm(p):
    p = np.asarray(p, dtype=float); out = np.full_like(p, np.nan)
    ok = ~np.isnan(p); q = p[ok]; m = len(q)
    if m == 0: return out
    idx = np.argsort(q)
    adj = np.maximum.accumulate((m - np.arange(m)) * q[idx])
    res = np.empty(m); res[idx] = np.minimum(adj, 1.0)
    out[ok] = res
    return out

def bh(p):
    p = np.asarray(p, dtype=float); out = np.full_like(p, np.nan)
    ok = ~np.isnan(p); q = p[ok]; m = len(q)
    if m == 0: return out
    idx = np.argsort(q)
    adj = q[idx] * m / np.arange(1, m + 1)
    adj = np.minimum.accumulate(adj[::-1])[::-1]
    res = np.empty(m); res[idx] = np.minimum(adj, 1.0)
    out[ok] = res
    return out

BERPASANGAN = ("V1", "V2b", "V3", "V4b")

def analisis_selisih(df, ref="P-C", metrik=("accuracy", "balanced_accuracy", "recall", "f1", "average_precision", "roc_auc"),
                     skenario=SEMUA_VARIASI, alpha=0.05):
    baris = []
    for (ds, model, cfg), g in df.groupby(["dataset", "model", "config"]):
        c = g[g.prosedur == ref].sort_values("split")
        if c.empty:
            continue
        for k in skenario:
            w = g[g.prosedur == k].sort_values("split")
            if w.empty:
                continue
            for m in metrik:
                a, b = c[m].to_numpy(), w[m].to_numpy()
                if k in BERPASANGAN:
                    assert len(a) == len(b), "Jumlah lipatan berbeda; pembagian tidak sejajar."
                    est, lo, hi, t, p = ttest_terkoreksi(b - a)
                    uji = "corrected resampled t-test (berpasangan)"
                else:
                    est, lo, hi, t, p = b.mean() - a.mean(), np.nan, np.nan, np.nan, np.nan
                    uji = "deskriptif (tidak berpasangan)"
                baris.append({"dataset": ds, "model": model, "config": cfg, "skenario": k, "metrik": m,
                              "selisih_rata2": est, "ci95_low": lo, "ci95_high": hi, "t": t,
                              "p_value": p, "uji": uji})
    out = pd.DataFrame(baris)
    # koreksi uji ganda: per konfigurasi dan per metrik
    out["p_holm"] = np.nan; out["p_bh"] = np.nan
    for _, idx in out.groupby(["config", "metrik"]).groups.items():
        out.loc[idx, "p_holm"] = holm(out.loc[idx, "p_value"])
        out.loc[idx, "p_bh"] = bh(out.loc[idx, "p_value"])
    def label(r):
        if np.isnan(r.p_value):
            return "deskriptif" if r.uji.startswith("deskriptif") else "tidak berbeda"
        if r.p_holm < alpha:
            return "lebih tinggi" if r.selisih_rata2 > 0 else "lebih rendah"
        return "tidak berbeda"
    out["kesimpulan_holm"] = out.apply(label, axis=1)
    return out

selisih = analisis_selisih(semua)
selisih.to_csv(os.path.join(OUT_DIR, "selisih_variasi_vs_pc.csv"), index=False)

tampil = selisih[selisih.metrik.isin(["f1", "average_precision"])].copy()
for kol in ["selisih_rata2", "ci95_low", "ci95_high", "t"]:
    tampil[kol] = tampil[kol].round(3)
for kol in ["p_value", "p_holm", "p_bh"]:
    tampil[kol] = tampil[kol].round(4)
tampil.drop(columns=["config", "uji"])

In [ ]:
# Ringkasan per dataset dan variasi: rata-rata selisih lintas model, dan jumlah model yang berbeda nyata (Holm)
f = selisih[selisih.metrik.isin(["f1", "average_precision", "accuracy", "roc_auc"])]
ring = f.groupby(["dataset", "skenario", "metrik"])["selisih_rata2"].mean().round(3).unstack("metrik")
nyata = (f[f.metrik == "f1"].assign(naik=lambda d: d.kesimpulan_holm == "lebih tinggi")
         .groupby(["dataset", "skenario"])["naik"].agg(lambda s: f"{s.sum()}/{len(s)}"))
ring["model_F1_naik_nyata(Holm)"] = nyata
print("Rata-rata selisih (variasi - P-C) lintas model:")
ring

### Dekomposisi selisih V2: kebocoran sungguhan vs efek komposisi data uji
`V2 - P-C` = `(V2b - P-C)` + `(V2 - V2b)`. Bagian pertama adalah kebocoran yang benar-benar membuat skor terlalu optimistis pada data asli. Bagian kedua muncul hanya karena data uji V2 berisi titik sintetis. Hal yang sama berlaku untuk V4 dan V4b.

In [ ]:
def dekomposisi(df_sel, total="V2", bersih="V2b", metrik=("f1", "average_precision", "roc_auc", "balanced_accuracy")):
    s = df_sel[df_sel.metrik.isin(metrik)].pivot_table(index=["dataset", "model", "metrik"],
                                                        columns="skenario", values="selisih_rata2")
    out = pd.DataFrame({f"total ({total}-P-C)": s[total],
                        f"kebocoran ({bersih}-P-C)": s[bersih],
                        f"komposisi uji ({total}-{bersih})": s[total] - s[bersih]})
    return out

def tambah_porsi(t):
    # porsi kebocoran = kebocoran / total; hanya dihitung bila total cukup besar (> 0,02) agar tidak menyesatkan
    t = t.copy()
    t["porsi_kebocoran_%"] = (100 * t.iloc[:, 1] / t.iloc[:, 0]).where(t.iloc[:, 0] > 0.02)
    return t.round(3)

dek_v2 = dekomposisi(selisih[selisih.config == "utama"], "V2", "V2b")
dek_v4 = dekomposisi(selisih[selisih.config == "utama"], "V4", "V4b")
ring_v2 = tambah_porsi(dek_v2.groupby(level=["dataset", "metrik"]).mean())   # porsi dihitung dari rata-rata selisih
ring_v4 = tambah_porsi(dek_v4.groupby(level=["dataset", "metrik"]).mean())
dek_v2, dek_v4 = tambah_porsi(dek_v2), tambah_porsi(dek_v4)
for nama_f, t in [("dekomposisi_V2_per_model", dek_v2), ("dekomposisi_V4_per_model", dek_v4),
                  ("dekomposisi_V2_ringkas", ring_v2), ("dekomposisi_V4_ringkas", ring_v4)]:
    t.to_csv(os.path.join(OUT_DIR, nama_f + ".csv"))
print("Dekomposisi V4 (rata-rata lintas model):")
display(ring_v4)
print("Dekomposisi V2 (rata-rata lintas model):")
ring_v2

In [ ]:
# Grafik 1: selisih F1 per variasi urutan praproses
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for ax, ds in zip(axes, DATASETS):
    s = selisih[(selisih.dataset == ds) & (selisih.metrik == "f1")]
    pv = s.pivot(index="skenario", columns="model", values="selisih_rata2").reindex(list(SEMUA_VARIASI))
    pv.plot.bar(ax=ax, rot=0)
    ax.axhline(0, color="k", lw=0.8)
    ax.set_title(f"{ds}: selisih F1 (variasi - P-C)")
    ax.set_xlabel("Variasi urutan praproses")
axes[0].set_ylabel("Selisih F1")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "grafik_selisih_f1.png"), dpi=150)
plt.show()

In [ ]:
# Grafik 2: sebaran F1 per pembagian untuk P-A/B/C dan V1-V4 (satu model, ubah MODEL_PLOT bila perlu)
MODEL_PLOT = "RF"
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for ax, ds in zip(axes, DATASETS):
    s = semua[(semua.dataset == ds) & (semua.model == MODEL_PLOT)]
    data = [s[s.prosedur == p]["f1"].to_numpy() for p in urutan]
    ax.boxplot(data)
    ax.set_xticklabels(urutan)
    ax.set_title(f"{ds} | {MODEL_PLOT}: sebaran F1 antar pembagian")
    ax.tick_params(axis="x", rotation=30)
axes[0].set_ylabel("F1")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "grafik_sebaran_f1.png"), dpi=150)
plt.show()

## Bagian 10. Langkah 13: Uji sensitivitas
Pertanyaannya: apakah **kesimpulan** (arah dan besar selisih) berubah jika seed atau metode imputasi diganti? Dijalankan untuk P-C, V1, V2, V2b, dan V3 (V4 dan V4b dilewati agar waktu proses wajar). Skenario penyeimbangan A/B/C sudah diperbandingkan di Bagian 6.

In [ ]:
KONFIG_SENS = [("seed=7", dict(seed=7)),
               ("seed=2024", dict(seed=2024)),
               ("imputasi=KNN", dict(imputer="knn"))]

SKEN_SENS = ("V1", "V2", "V2b", "V3")
hasil_sens = []
for nama_cfg, kw in KONFIG_SENS:
    print("=== Konfigurasi sensitivitas:", nama_cfg)
    b = jalankan_semua_pipa(models=SENS_MODELS, config=nama_cfg, balances=("smote",), **kw)
    k = jalankan_semua_variasi(models=SENS_MODELS, skenario=SKEN_SENS, config=nama_cfg, **kw)
    hasil_sens += [b, k]

sens = pd.concat([semua[(semua.model.isin(SENS_MODELS)) & (semua.prosedur.isin(["P-C", *SKEN_SENS]))]]
                 + hasil_sens, ignore_index=True)
sens.to_csv(os.path.join(OUT_DIR, "hasil_sensitivitas_mentah.csv"), index=False)

sel_sens = analisis_selisih(sens, skenario=SKEN_SENS)
sel_sens.to_csv(os.path.join(OUT_DIR, "selisih_sensitivitas.csv"), index=False)

tbl = (sel_sens[sel_sens.metrik == "f1"]
       .groupby(["dataset", "skenario", "config"])["selisih_rata2"].mean().round(3)
       .unstack("config"))
print("Rata-rata selisih F1 (variasi - P-C) lintas model, per konfigurasi:")
tbl

In [ ]:
# Apakah kesimpulan sama di semua konfigurasi?
# Untuk variasi berpasangan dipakai kesimpulan Holm per model; untuk V2 dipakai tanda rata-rata selisih.
def kesimpulan_cfg(g):
    if g.uji.iloc[0].startswith("deskriptif"):
        return "lebih tinggi" if g.selisih_rata2.mean() > 0 else "lebih rendah"
    hit = g.kesimpulan_holm.value_counts()
    return f"{hit.get('lebih tinggi', 0)} naik / {hit.get('lebih rendah', 0)} turun / {hit.get('tidak berbeda', 0)} sama"

kes = (sel_sens[sel_sens.metrik == "f1"].groupby(["dataset", "skenario", "config"])
       .apply(kesimpulan_cfg).unstack("config"))
kes["konsisten"] = kes.nunique(axis=1) == 1
print("Kesimpulan F1 per konfigurasi (jumlah model; uji berpasangan, koreksi Holm):")
kes

## Bagian 11. Langkah 14: Dokumentasi dan reproduksibilitas
Sel di bawah menyimpan versi pustaka, konfigurasi, dan `requirements.txt`. Sertakan file ini, folder `results/`, dan notebook saat mengunggah ke GitHub atau Zenodo.

In [ ]:
import os, json, platform, datetime
from importlib.metadata import version, PackageNotFoundError

OUT_DIR = globals().get("OUT_DIR", "results")
os.makedirs(OUT_DIR, exist_ok=True)

PAKET = ["numpy", "pandas", "scipy", "scikit-learn", "imbalanced-learn", "xgboost", "matplotlib"]
versi = {}
for p in PAKET:
    try:
        versi[p] = version(p)
    except PackageNotFoundError:
        versi[p] = "tidak ditemukan"

with open(os.path.join(OUT_DIR, "requirements.txt"), "w") as f:
    for p, v in versi.items():
        f.write(f"{p}=={v}\n")

def ambil(nama):
    return globals().get(nama, "tidak tersedia")

konfig = {"python": platform.python_version(), "os": platform.platform(),
          "tanggal_jalan": str(datetime.date.today()),
          "FAST_MODE": ambil("FAST_MODE"), "SEED": ambil("SEED"),
          "N_SPLITS": ambil("N_SPLITS"), "N_REPEATS": ambil("N_REPEATS"),
          "INNER_FOLDS": ambil("INNER_FOLDS"), "K_FEATURES": ambil("K_FEATURES"),
          "N_TREES": ambil("N_TREES"), "TANGGAL_AKSES_DATA": ambil("TANGGAL_AKSES"),
          "versi_paket": versi}

with open(os.path.join(OUT_DIR, "konfigurasi_eksperimen.json"), "w") as f:
    json.dump(konfig, f, indent=2, default=str)
konfig

## Panduan menulis hasil (agar apa pun hasilnya jelas)
1. **Laporkan semua angka**, termasuk selisih yang kecil atau nol. Selisih nol bukan kegagalan, itu temuan.
2. **Pisahkan V2 menjadi dua angka:** kebocoran sungguhan (V2b - P-C) dan efek komposisi data uji (V2 - V2b). Jangan menulis seluruh selisih V2 sebagai "kebocoran".
3. **Uji statistik:** sebut *corrected resampled t-test* dengan koreksi Holm. Selisih yang tidak berbeda nyata ditulis "tidak berbeda", bukan "tidak konsisten".
4. **Gunakan kata yang hati-hati:** tulis "variasi V2 memberi F1 rata-rata X poin lebih tinggi daripada P-C pada dataset ini", bukan penilaian terhadap studi lain.
5. **Sebutkan keterbatasan:** jumlah kasus positif kecil, *corrected t-test* masih berupa pendekatan, V2 dan V4 tidak berpasangan, proporsi titik sintetis di data latih V2 (~80%) sedikit berbeda dari V2b (100%), data donor darah dan data tahun 1980-an tidak mewakili populasi klinis saat ini, dan tidak ada validasi eksternal.
6. **Bandingkan dengan baseline mayoritas** setiap kali menyebut akurasi.
7. **Sertakan** `results/`, `requirements.txt`, `konfigurasi_eksperimen.json`, dan notebook ini sebagai lampiran replikasi.